# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [39]:
#Load Dataset
df = pd.read_csv("data/AviationData.csv", encoding="latin-1")

#Preview data
display(df.head())

#Inspect datatypes and NaNs
df.info()

#Count NaNs
display(df.isna().sum())

#Summary statistics
display(df.describe())

/tmp/ipykernel_1919/567694012.py:2: DtypeWarning: Columns (6: Latitude, 7: Longitude, 28: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/AviationData.csv", encoding="latin-1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [40]:
# Inspect relevant columns
display(df[["Aircraft.Category", "Amateur.Built", "Event.Date"]].head())
print(df[["Aircraft.Category", "Amateur.Built", "Event.Date"]].isna().sum())

# Convert Event.Date to datetime
df["Event.Date"] = pd.to_datetime(df["Event.Date"])

# Filter for airplanes, professional builds, and events from 1983 onward
df = df[
    (df["Aircraft.Category"] == "Airplane") &
    (df["Amateur.Built"] == "No") &
    (df["Event.Date"].dt.year >= 1983)
].copy()

# Check the filtered dataset
print("Rows remaining:", len(df))
display(df.head())


,Aircraft.Category,Amateur.Built,Event.Date
0,NaN,No,1948-10-24
1,NaN,No,1962-07-19
2,NaN,No,1974-08-30
3,NaN,No,1977-06-19
4,NaN,No,1979-08-02


Aircraft.Category    56602
Amateur.Built          102
Event.Date               0
dtype: int64
Rows remaining: 21447


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,Unknown,NaN,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,Unknown,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [41]:


display(df[["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]].head())
print(df[["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]].isna().sum())

#missing injury values are assumed to represent zero reported injuries
# in that injury category, so they are filled with zero
injury_cols =[
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

df[injury_cols] = df[injury_cols].fillna(0)

# total number of people involved in each accident
df["Total.People"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"] + df["Total.Minor.Injuries"] + df["Total.Uninjured"]

# Display the injury columns and calculated total
display(df[[
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Total.People"
]].head())

# Calculate serious/fatal injury rate
df["Serious.Fatal.Rate"] = (
    (df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"])
    / df["Total.People"]
)

# Rows with zero recorded people cannot produce a meaningful injury rate.
# Their Serious.Fatal.Rate is left as NaN rather than assuming a 0% injury rate.
print("Rows with zero total people:", (df["Total.People"] == 0).sum())
print("Missing injury rates:", df["Serious.Fatal.Rate"].isna().sum())


display(df[[
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Total.People",
    "Serious.Fatal.Rate"
]].head())

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
4149,NaN,NaN,NaN,588.0
4150,NaN,NaN,NaN,588.0
4171,1.0,1.0,NaN,NaN
4285,1.0,NaN,NaN,4.0
5957,NaN,NaN,NaN,289.0


Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.People
4149,0.0,0.0,0.0,588.0,588.0
4150,0.0,0.0,0.0,588.0,588.0
4171,1.0,1.0,0.0,0.0,2.0
4285,1.0,0.0,0.0,4.0,5.0
5957,0.0,0.0,0.0,289.0,289.0


Rows with zero total people: 904
Missing injury rates: 904


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.People,Serious.Fatal.Rate
4149,0.0,0.0,0.0,588.0,588.0,0.0
4150,0.0,0.0,0.0,588.0,588.0,0.0
4171,1.0,1.0,0.0,0.0,2.0,1.0
4285,1.0,0.0,0.0,4.0,5.0,0.2
5957,0.0,0.0,0.0,289.0,289.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [42]:
display(df["Aircraft.damage"].value_counts(dropna=False))

# Track whether the aircraft was destroyed.
# Unknown and missing damage values remain NaN rather than being assumed not destroyed.
df["Destroyed"] = np.where(
    df["Aircraft.damage"] == "Destroyed",
    1,
    np.where(
        df["Aircraft.damage"].isin(["Substantial", "Minor"]),
        0,
        np.nan
    )
)

display(df[["Aircraft.damage", "Destroyed"]].head())

print(df["Destroyed"].value_counts(dropna=False))

Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64

,Aircraft.damage,Destroyed
4149,Minor,0.0
4150,Minor,0.0
4171,Destroyed,1.0
4285,NaN,NaN
5957,Minor,0.0


Destroyed
0.0    17807
1.0     2316
NaN     1324
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [43]:
# Check missing Make values
print("Missing Make values:", df["Make"].isna().sum())

# Inspect Make names and their counts
display(df["Make"].value_counts(dropna=False))

#standardize makes
df["Make"] = df["Make"].str.strip().str.upper()

# new list, valid makes
# valid makes with 50 or more
make_counts = df["Make"].value_counts()
valid_makes = make_counts[make_counts >= 50].index

#checking valid makes is in df
df = df[df["Make"].isin(valid_makes)].copy()

# Verify cleaning and 50-record threshold
print("Rows remaining:", len(df))
print("Number of valid makes:", df["Make"].nunique())
display(df["Make"].value_counts())

Missing Make values: 3


Make
CESSNA               4867
PIPER                2803
Cessna               2279
Piper                1186
BOEING               1037
                     ... 
BELL                    1
DETRICK DONALD G        1
PHANTOM                 1
JAMES R DERNOVSEK       1
ORLICAN S R O           1
Name: count, Length: 1333, dtype: int64

Rows remaining: 17892
Number of valid makes: 36


Make
CESSNA                            7146
PIPER                             3989
BEECH                             1431
BOEING                            1264
MOONEY                             363
AIRBUS                             243
CIRRUS DESIGN CORP                 220
BELLANCA                           219
AIR TRACTOR INC                    219
MAULE                              215
AIR TRACTOR                        206
AERONCA                            200
CHAMPION                           158
EMBRAER                            153
GRUMMAN                            147
LUSCOMBE                           141
CIRRUS                             137
STINSON                            129
MCDONNELL DOUGLAS                  108
NORTH AMERICAN                     106
DEHAVILLAND                         95
TAYLORCRAFT                         93
AERO COMMANDER                      90
AVIAT AIRCRAFT INC                  76
SOCATA                              75
DIAMOND AIRCRAFT IND

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [44]:
# Check and remove missing Model values
print("Missing Model values:", df["Model"].isna().sum())

df = df.dropna(subset=["Model"]).copy()

print("Missing Model values after cleaning:", df["Model"].isna().sum())

# Inspect counts for each Make/Model combination
display(df[["Make", "Model"]].value_counts())

# Create a unique plane type identifier using Make and Model
df["Plane.Type"] = df["Make"] + " " + df["Model"]

display(df[["Make", "Model", "Plane.Type"]].head())


Missing Model values: 13
Missing Model values after cleaning: 0


Make            Model      
CESSNA          172            769
BOEING          737            403
CESSNA          152            316
                182            304
                172S           276
                              ... 
                T206G            1
BOMBARDIER INC  BD-700-2A12      1
BOEING          737-8            1
CESSNA          P206D            1
PIPER           PA-44            1
Name: count, Length: 2159, dtype: int64

,Make,Model,Plane.Type
4150,BOEING,747,BOEING 747
4171,PIPER,PA-28-140,PIPER PA-28-140
4285,DE HAVILLAND,DHC-6,DE HAVILLAND DHC-6
6760,BOEING,727-200,BOEING 727-200
6806,BEECH,C35,BEECH C35


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [45]:
columns_to_inspect = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for col in columns_to_inspect:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False))

# Clean unknown Engine.Type values
df["Engine.Type"] = df["Engine.Type"].replace(
    ["Unknown", "UNK"],
    np.nan
)

# Clean unknown Weather.Condition values
df["Weather.Condition"] = df["Weather.Condition"].replace(
    ["Unk", "UNK"],
    np.nan
)

df["Number.of.Engines"] = df["Number.of.Engines"].replace(
    0,
    np.nan
)

# Clean Purpose.of.flight
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace({
    "Unknown": np.nan,
    "Air Race/show": "Air Race show"
})

# Treat unknown flight phase as missing
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].replace(
    "Unknown",
    np.nan
)


--- Engine.Type ---


Engine.Type
Reciprocating      12835
NaN                 3214
Turbo Prop           931
Turbo Fan            701
Unknown              105
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64


--- Weather.Condition ---


Weather.Condition
VMC    14295
NaN     2417
IMC      905
Unk      186
UNK       76
Name: count, dtype: int64


--- Number.of.Engines ---


Number.of.Engines
1.0    13222
2.0     2470
NaN     2089
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64


--- Purpose.of.flight ---


Purpose.of.flight
Personal                     9844
NaN                          3047
Instructional                2410
Aerial Application            724
Business                      409
Unknown                       303
Positioning                   269
Skydiving                     157
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry                          72
Executive/corporate            65
Glider Tow                     29
Public Aircraft - Federal      28
Public Aircraft                27
Public Aircraft - State        21
Air Race show                  15
Firefighting                   12
Public Aircraft - Local        11
PUBS                            3
Air Race/show                   2
Air Drop                        2
ASHO                            2
Name: count, dtype: int64


--- Broad.phase.of.flight ---


Broad.phase.of.flight
NaN            15427
Landing         1110
Takeoff          425
Cruise           238
Approach         210
Maneuvering      127
Taxi              99
Go-around         81
Descent           62
Climb             52
Standing          35
Unknown           11
Other              2
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [47]:
# Inspect non-null counts for each column
non_null_counts = df.notna().sum()
display(non_null_counts)

# Check current dataframe size
print("Current dataframe shape:", df.shape)

# After the required row filtering in earlier steps, the dataframe
# contains fewer than 20,000 rows. Therefore, no remaining column
# can have more than 20,000 non-null values, so applying that threshold
# would remove every column.

Event.Id                  17879
Investigation.Type        17879
Accident.Number           17879
Event.Date                17879
Location                  17875
Country                   17878
Latitude                  15981
Longitude                 15978
Airport.Code              11648
Airport.Name              11754
Injury.Severity           17162
Aircraft.damage           16815
Aircraft.Category         17879
Registration.Number       17715
Make                      17879
Model                     17879
Amateur.Built             17879
Number.of.Engines         15785
Engine.Type               14559
FAR.Description           17535
Schedule                   2139
Purpose.of.flight         14529
Air.carrier                8448
Total.Fatal.Injuries      17879
Total.Serious.Injuries    17879
Total.Minor.Injuries      17879
Total.Uninjured           17879
Weather.Condition         15200
Broad.phase.of.flight      2441
Report.Status             14094
Publication.Date          17091
Total.Pe

Current dataframe shape: (17879, 35)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [48]:
# Save cleaned dataframe for use in the analysis notebook
df.to_csv("data/AviationData_Clean.csv", index=False)

print("Cleaned data saved successfully.")

Cleaned data saved successfully.
